<H1>Assignment 7: Logistic Regression</H1>

<H2>Objective:</H2> 
The goal of this assignment is to apply Logistic Regression to the iris dataset for classification tasks using two different libraries: scikit-learn and PyMC. Through this exercise, students will gain practical experience in implementing Logistic Regression models, understand the differences between using a traditional machine learning library (scikit-learn) and a probabilistic programming framework (PyMC), and explore the logistic function in the context of Logistic Regression.

<b>Task 1: Logistic Regression using scikit-learn</b> 

<ol>
  <li>Load the Iris dataset from scikit-learn.
</li>
  <li>Split the dataset into training and testing sets (80% training, 20% testing).
</li>
  <li>Use LogisticRegression from sklearn.linear_model to train a model on the training set. Note: Use multi_class='multinomial' and solver='lbfgs' to enable multinomial logistic regression.
</li>
  <li>Evaluate the model on the testing set and report the accuracy. In addition, evaluate the model with the second feature only. 
</li>   
</ol>

In [8]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(solver='lbfgs', max_iter=200) #multi_class='multinomial' funktioniert nicht
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy_all = accuracy_score(y_test, y_pred)
print(f"Task 1 - Accuracy with all features: {accuracy_all:.4f}")

X_train_feat2 = X_train[:, 1].reshape(-1, 1)
X_test_feat2 = X_test[:, 1].reshape(-1, 1)
model_feat2 = LogisticRegression(solver='lbfgs', max_iter=200)
model_feat2.fit(X_train_feat2, y_train)
y_pred_feat2 = model_feat2.predict(X_test_feat2)
accuracy_feat2 = accuracy_score(y_test, y_pred_feat2)
print(f"Task 1 - Accuracy with only feature 2: {accuracy_feat2:.4f}")


Task 1 - Accuracy with all features: 1.0000
Task 1 - Accuracy with only feature 2: 0.6000


<b>Task 2: Logistic Regression using PyMC</b> 

You will utilize PyMC for model building and inference, and assess the model's performance using accuracy metrics. This assignment will help you understand the process of defining a Bayesian model, performing posterior inference, and making predictions based on the Bayes estimator (posterior mean) instead of full posterior predictive distribution.
<ol>
  <li>Load the Iris dataset from scikit-learn.
</li>
  <li>Split the dataset into training and testing sets (80% training, 20% testing).
</li>
  <li> Convert the target variable into a binary classification problem (setosa vs. not-setosa).
</li>
      <li> Implement the logistic regression model in PyMC and evaluate accuracy.
</li>
</ol>

<b>Model Background:</b>

Define a Bayesian logistic regression model with priors for the intercept (\alpha) and coefficient (\beta). The logistic function will transform the linear combination of inputs ($X\beta + \alpha$) into probabilities.

<b>Priors:</b>

Priors are specified for the intercept and coefficients to reflect our beliefs about their values before observing the data. Normally distributed priors are commonly used.
<ol>
  <li>Prior Intercept ($\alpha$): $\alpha \sim \text{Normal}(0, 10)$
</li>
  <li>Prior Coefficients ($\beta$): 
$\beta \sim \text{Normal}(0, 10)$</li>

</ol>

<b>Likelihood:</b>

The likelihood represents how probable the observed data is, given the parameters of the model. For binary classification tasks, a Bernoulli distribution is appropriate.
<ol>
    <li>Logit Model (pymc deterministic): $z=X\beta + \alpha$ </li>
  <li>Observed data likelihood: Given the data matrix $X$, we model the likelihood of observing $Y$ given the parameters assuming a Bernoulli distribution: $Y \sim \text{Bernoulli}(p)$, where $p = \frac{1}{1 + e^{-(X\beta + \alpha)}}$. The logic function (sigmoid) is given $\sigma(z)=\frac{1}{1 + e^{-z}}$
</li>
</ol>

<b>Posterior Inference and Predictions:</b>

After defining the model, we perform posterior inference to update our beliefs about the parameters ($\alpha,\beta) based on the observed data. This is typically done through Markov Chain Monte Carlo (MCMC) sampling.

Posterior predictions involve computing the linear combination of the input features and the posterior mean estimates of the model parameters, then transforming these logits into probabilities using the logistic function. Predictions are made based on whether these probabilities exceed a certain threshold (e.g., 0.5 for binary classification).

<b>Hints and Summary:</b>

Pay close attention to the shape and dimensions of your data, especially when performing matrix operations.
<ol>
    <li>Use PyMC to define this model and sample from the posterior</li>
  <li>Extract the mean of the posterior distributions for $\alpha$ and $\beta$.</li>
   <li>Use these posterior means to compute logits for the test set: $z=X_{test}\beta + \alpha$  </li>
   <li>Transform logits into probabilities using the logistic function.</li>
    <li>Make predictions based on a probability threshold of 0.5.</li>
</ol>

Take a look at: <url>https://docs.xarray.dev/en/stable/generated/xarray.DataArray.expand_dims.html</url> and <url>https://python.arviz.org/en/stable/getting_started/XarrayforArviZ.html#xarray-for-arviz</url>

In [10]:

import pymc as pm
import arviz as az

y_binary = (iris.target == 0).astype(int)
X_data = iris.data

# Split data
X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(
    X_data, y_binary, test_size=0.2, random_state=42
)

#1. define Model
with pm.Model() as logistic_model:
    # Priors
    alpha = pm.Normal('intercept', mu=0, sigma=10)
    beta = pm.Normal('coefficients', mu=0, sigma=10, shape=X_train_bin.shape[1])

    z = pm.math.dot(X_train_bin, beta) + alpha

    p = pm.math.sigmoid(z)

    y_obs = pm.Bernoulli('y_obs', p=p, observed=y_train_bin)

    trace = pm.sample(2000, tune=1000, random_seed=42, progressbar=True)


# 2. Extract means
posterior_samples = trace.posterior
alpha_mean = float(posterior_samples['intercept'].mean())
beta_mean = posterior_samples['coefficients'].mean(dim=['chain', 'draw']).values

# 3. Compute logits
z_test =np.dot(X_test_bin, beta_mean) + alpha_mean
# 4. use logistic function
p_test = 1 / (1 + np.exp(-z_test))
# 5. Make predictions threshold 0.5
y_pred_pymc = (p_test > 0.5).astype(int)

# Calculate accuracy
accuracy_pymc = accuracy_score(y_test_bin, y_pred_pymc)
print(f"Task 2 - PyMC Accuracy (Binary: Setosa vs. Not-Setosa): {accuracy_pymc:.4f}")

# Print model summary
print("\nPyMC Model Summary:")
print(az.summary(trace, var_names=['intercept', 'coefficients']))



Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [intercept, coefficients]


C:\Users\henri\miniconda3\envs\gmlp\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 7 seconds.
There were 2476 divergences after tuning. Increase `target_accept` or reparameterize.


Task 2 - PyMC Accuracy (Binary: Setosa vs. Not-Setosa): 1.0000

PyMC Model Summary:
                mean   sd eti89_lb eti89_ub  ess_bulk  ess_tail r_hat  \
intercept        1.4  9.4      -13       17      1608      1859  1.00   
coefficients[0]  2.2  5.1     -5.8       10      1108      1512  1.00   
coefficients[1]  7.6  6.9     -2.8       19      1198      2131  1.00   
coefficients[2]  -12  6.3      -23     -2.5      1083      1780  1.00   
coefficients[3]   -6    9      -20      8.3      1767      2151  1.00   

                mcse_mean mcse_sd  
intercept            0.24    0.16  
coefficients[0]      0.15    0.11  
coefficients[1]       0.2    0.15  
coefficients[2]      0.19    0.14  
coefficients[3]      0.22    0.17  
